# Introduction

This notebook validates the `RiskManager` with real data from the `PricingEngine`
and `LucicTseQuotingEngine`. It demonstrates:
- Normal operation (no breaches)
- Delta breach with excess reporting
- Gamma breach
- Multiple breaches
- Drawdown breach and reset

# Import Libraries

In [1]:
import sys
from pathlib import Path
import logging
import time
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

from src.pricing_engine import PricingEngine
from src.quoting.lucic_tse import LucicTseQuotingEngine
from src.risk.risk_manager import RiskManager

# Setup

## Initialize Pricing Engine

In [2]:
SYMBOL = "SPY"
R = 0.045
Q = 0.012
INITIAL_CAPITAL = 100000.0

# 1. Create and calibrate PricingEngine
pricing_engine = PricingEngine(symbol=SYMBOL, r=R, q=Q, max_expiries=4)
pricing_engine.start_background_calibration(interval_ms=2000)

print("Waiting for PricingEngine calibration...")
while not pricing_engine.is_calibrated():
    time.sleep(0.5)
print("PricingEngine calibrated.")

spot = pricing_engine.get_spot()
print(f"Spot: {spot:.2f}")

2026-07-13 13:08:24,218 - src.pricing_engine - INFO - Starting initial calibration...
2026-07-13 13:08:30,530 - src.pricing_engine - INFO - Initial calibration completed successfully.
2026-07-13 13:08:30,532 - src.pricing_engine - INFO - Background calibration thread started (interval=2000ms).


Waiting for PricingEngine calibration...
PricingEngine calibrated.
Spot: 754.95


## SVI Calibration

In [3]:
svi_slices = pricing_engine.get_svi_params()
expiries = sorted(svi_slices.keys())[:2]
option_specs = []
for expiry in expiries:
    T = svi_slices[expiry]['T']
    strikes = [int(spot * (1 + i*0.01)) for i in range(-3, 4)]
    for strike in strikes:
        option_specs.append({
            'id': f"SPY_{strike}_{expiry}",
            'strike': float(strike),
            'expiry': expiry,
            'T': T,
            'option_type': 'call',
        })
print(f"Created {len(option_specs)} option specs.")

Created 14 option specs.


c:\Users\Danny\Desktop\Stuff\Projects\Projects - Finance\Market Making Bot\Volatility-Adaptive-Options-Market-Making-Engine\src\models\svi.py:13: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(svi_total_variance(k, a, b, rho, m, sigma) / T)
2026-07-13 13:09:08,072 - src.pricing_engine - WARNING - 2026-07-13: theta = -0.005215 <= 0. Skipping slice.


## Define risk factors and order flow parameters

In [4]:
risk_factors = [
    {
        'name': 'Vega 0-7D',
        'alpha': 10.0,
        'membership': lambda spec: spec['T'] <= 7/365,
        'weight_key': 'vega',
    },
    {
        'name': 'Vega 8-30D',
        'alpha': 5.0,
        'membership': lambda spec: 7/365 < spec['T'] <= 30/365,
        'weight_key': 'vega',
    },
    {
        'name': 'Total Delta',
        'alpha': 1.0,
        'membership': lambda spec: True,
        'weight_key': 'delta',
    },
]
order_flow_params = {
    'lambda0_a': 50 * 252,
    'lambda0_b': 50 * 252,
    'kappa_a': 0.75,
    'kappa_b': 0.75,
}

## Quoting Engine

In [5]:
quoting_engine = LucicTseQuotingEngine(
    pricing_engine=pricing_engine,
    risk_factors=risk_factors,
    order_flow_params=order_flow_params,
    horizon_hours=0.5,
    risk_aversion=0.1,
    auto_update=True,
    update_interval_ms=500,
    initial_realized_vol=0.18,
    option_specs=option_specs,
)

print("Waiting for quoting engine initial state...")
while not quoting_engine.is_initialized():
    time.sleep(0.5)
print("Quoting engine initialized.")


2026-07-13 13:10:55,701 - src.quoting.lucic_tse - INFO - Initial realized vol set to 0.1800
2026-07-13 13:10:55,703 - src.quoting.lucic_tse - INFO - Initialised with 14 option specs.
2026-07-13 13:10:55,703 - src.quoting.lucic_tse - INFO - Performing initial synchronous state update...
2026-07-13 13:10:55,808 - src.quoting.lucic_tse - INFO - Initial state update completed.
2026-07-13 13:10:55,810 - src.quoting.lucic_tse - INFO - Starting background update thread (interval=500ms).


Waiting for quoting engine initial state...
Quoting engine initialized.


## Finally: Risk Manager

In [6]:
risk_manager = RiskManager(
    quoting_engine=quoting_engine,
    delta_limit=50000.0,
    gamma_limit=-100.0,
    theta_limit=-100.0,
    vega_limits={"0-7D": 500.0, "8-30D": 1500.0},
    drawdown_limit=-0.02,
    initial_capital=INITIAL_CAPITAL,
)
print("Risk Manager initialized.")

Risk Manager initialized.


# Test: Risk Manager

## Scenario 1: Normal Operation (No Breaches)

In [10]:
print("=" * 60)
print("SCENARIO 1: Normal Operation")
print("=" * 60)

positions = {}  # No inventory
result = risk_manager.get_quotes(option_specs, positions, update_pnl=0.0)

if result.halted:
    print(f"❌ Halted unexpectedly: {result.reason}")
else:
    print(f"✅ Quotes generated: {len(result.quotes)} options")
    sample = list(result.quotes.values())[0]
    print(f"Sample quote: {sample.option_id} Bid={sample.bid:.2f} Ask={sample.ask:.2f} Size={sample.bid_size}/{sample.ask_size}")

SCENARIO 1: Normal Operation
✅ Quotes generated: 14 options
Sample quote: SPY_732_2026-07-13 Bid=26.60 Ask=29.42 Size=1/1


## Delta Breach

In [ ]:
print("=" * 60)
print("SCENARIO 2: Delta Breach")
print("=" * 60)

# Set position
positions_high = {option_specs[0]['id']: 1000}  # 1000 calls, each ~0.5 delta (ATM) = 500 Delta (very small)

print("Lowering delta limit to 100 for demonstration...")
risk_manager.set_delta_limit(100.0)

# Add a position that pushes Delta to ~150
positions_high = {option_specs[0]['id']: 300}  # Roughly 150 Delta (0.5 * 300)

result = risk_manager.get_quotes(option_specs, positions_high, update_pnl=0.0)

if result.halted:
    print(f"✅ Delta breach detected!")
    print(f"   Reason: {result.reason}")
    print(f"   Excess Delta: {result.excess_delta:.2f}")
else:
    print("❌ Delta breach not detected (unexpected).")

# Reset delta limit
risk_manager.set_delta_limit(50000.0)

SCENARIO 2: Delta Breach
Lowering delta limit to 100 for demonstration...
✅ Delta breach detected!
   Reason: Delta (excess 123.57); Theta (excess 6860.99); Vega {'0-7D': np.float64(1811.2433695663276)}
   Excess Delta: 123.57


# Cleanup

In [13]:
# --- Cleanup ---
print("\n" + "=" * 60)
print("CLEANUP")
print("=" * 60)

quoting_engine.stop_background_update()
pricing_engine.stop_background_calibration()
print("Cleanup complete.")

2026-07-13 13:14:06,216 - src.quoting.lucic_tse - INFO - Stopping background update thread...
2026-07-13 13:14:06,281 - src.quoting.lucic_tse - INFO - Background update thread stopped (or marked for stopping).
2026-07-13 13:14:06,282 - src.pricing_engine - INFO - Stopping background calibration thread...



CLEANUP


2026-07-13 13:14:07,749 - src.pricing_engine - INFO - Background calibration thread stopped.


Cleanup complete.
